# 02 Analysis

This notebook uses the processed data and summary CSV files to reproduce the core analysis behind the interactive report in `docs/index.html`.


In [ ]:
from functools import lru_cache
import json
from pathlib import Path
import subprocess
import sys
from typing import Dict

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO_URL = "https://github.com/lexieliujy/POLI3148_PS1.git"


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "docs" / "index.html").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root folder.")


try:
    ROOT_DIR = find_project_root(Path.cwd())
except FileNotFoundError:
    ROOT_DIR = Path.cwd() / "POLI3148_PS1"
    if not (ROOT_DIR / "data").exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT_DIR)], check=True)

CODE_DIR = ROOT_DIR / "code"
DATA_DIR = ROOT_DIR / "data"
BOUNDARY_DATA_PATH = DATA_DIR / "support" / "ne_50m_admin_0_countries.zip"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

processed = pd.read_csv(DATA_DIR / "acled_mena_processed.csv", parse_dates=["event_date"])
yearly = pd.read_csv(DATA_DIR / "yearly_shift_summary.csv")
spatial = pd.read_csv(DATA_DIR / "spatial_bucket_summary.csv")
country = pd.read_csv(DATA_DIR / "country_pattern_summary.csv")

if processed.empty:
    raise RuntimeError("Processed data loaded as empty; check the data files.")

report_df = processed[processed["year"].between(1997, 2024)].copy()
report_df.shape


## Analysis helper functions

I define the figure helpers inside this notebook so the analysis is readable without importing the shared `project_utils.py` file. These functions use the processed event-level data and summary CSVs created in `01_data_cleaning.ipynb`.


In [ ]:
PALETTE = {
    "capital_remote": "#c44536",
    "border_battles": "#1f4e79",
    "remote_total": "#d98b5f",
    "other": "#6c7a89",
    "civilians": "#d8a47f",
}

CAPITALS_BY_COUNTRY: Dict[str, str] = {
    "Algeria": "Algiers",
    "Bahrain": "Manama",
    "Egypt": "Cairo",
    "Iran": "Tehran",
    "Iraq": "Baghdad",
    "Israel": "Jerusalem",
    "Jordan": "Amman",
    "Kuwait": "Kuwait City",
    "Lebanon": "Beirut",
    "Libya": "Tripoli",
    "Morocco": "Rabat",
    "Oman": "Muscat",
    "Saudi Arabia": "Riyadh",
    "Sudan": "Khartoum",
    "Syria": "Damascus",
    "Turkey": "Ankara",
    "Tunisia": "Tunis",
    "United Arab Emirates": "Abu Dhabi",
    "Yemen": "Sanaa",
}


@lru_cache(maxsize=1)
def load_world_geojson() -> dict:
    import geopandas as gpd

    world = gpd.read_file(f"zip://{BOUNDARY_DATA_PATH}")[["ADMIN", "geometry"]].copy()
    world = world.rename(columns={"ADMIN": "country_name"})
    return json.loads(world.to_json())


In [ ]:
def build_trend_figure(yearly: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["border_battle_events"],
            mode="lines+markers",
            name="Border-proximate battles",
            line={"color": PALETTE["border_battles"], "width": 3},
            marker={"size": 6},
            hovertemplate="Year %{x}<br>Events %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["capital_remote_events"],
            mode="lines+markers",
            name="Capital-area remote attacks",
            line={"color": PALETTE["capital_remote"], "width": 3},
            marker={"size": 6},
            hovertemplate="Year %{x}<br>Events %{y}<extra></extra>",
        )
    )
    fig.update_layout(
        title="Border-proximate battles remain more common than capital-area remote attacks",
        template="plotly_white",
        height=430,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        xaxis_title="Year",
        yaxis_title="Recorded events",
    )
    return fig


def build_remote_growth_figure(yearly: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=yearly["year"],
            y=yearly["drone_events"],
            name="Air/drone strikes",
            marker_color=PALETTE["capital_remote"],
            opacity=0.82,
            hovertemplate="Year %{x}<br>Drone events %{y}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["all_remote_events"],
            mode="lines+markers",
            name="All remote violence events",
            line={"color": PALETTE["border_battles"], "width": 3},
            marker={"size": 6},
            hovertemplate="Year %{x}<br>Remote events %{y}<extra></extra>",
        )
    )
    fig.update_layout(
        title="Drone strikes and remote violence both rise sharply over time in MENA",
        template="plotly_white",
        height=430,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        xaxis_title="Year",
        yaxis_title="Recorded events",
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    )
    return fig


def build_share_figure(yearly: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["border_share_of_battles_pct"],
            name="Border-proximate share of all battles",
            mode="lines+markers",
            line={"color": PALETTE["border_battles"], "width": 3},
            marker={"size": 7},
            hovertemplate="Year %{x}<br>%{y}% of battles<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=yearly["year"],
            y=yearly["capital_share_of_remote_pct"],
            mode="lines+markers",
            name="Capital-area share of all remote attacks",
            line={"color": PALETTE["capital_remote"], "width": 3},
            marker={"size": 7},
            hovertemplate="Year %{x}<br>%{y}% of remote events<extra></extra>",
        )
    )
    fig.update_layout(
        title="Neither share shows a clean shift from border conflict to capital targeting",
        template="plotly_white",
        height=430,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        xaxis_title="Year",
        yaxis={"title": "Share (%)"},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    )
    return fig


def build_spatial_composition_figure(df: pd.DataFrame) -> go.Figure:
    composition = (
        pd.crosstab(df["spatial_bucket"], df["event_type"], normalize="index")
        .mul(100)
        .reset_index()
        .melt(id_vars="spatial_bucket", var_name="event_type", value_name="share_pct")
    )
    fig = px.bar(
        composition,
        x="spatial_bucket",
        y="share_pct",
        color="event_type",
        barmode="stack",
        category_orders={
            "spatial_bucket": ["Capital areas", "Border-proximate areas", "Other areas"]
        },
        color_discrete_map={
            "Battles": PALETTE["border_battles"],
            "Explosions/Remote violence": PALETTE["capital_remote"],
            "Violence against civilians": PALETTE["civilians"],
        },
        labels={
            "spatial_bucket": "",
            "share_pct": "Share of events (%)",
            "event_type": "Event type",
        },
        title="Capital areas are more remote than the rest of the region, but not dominant",
    )
    fig.update_layout(
        template="plotly_white",
        height=420,
        margin={"l": 40, "r": 20, "t": 70, "b": 45},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    )
    fig.update_traces(hovertemplate="%{x}<br>%{fullData.name}: %{y:.1f}%<extra></extra>")
    return fig


In [ ]:
def build_geography_map_figure(df: pd.DataFrame) -> go.Figure:
    max_points_per_category = 18000
    plot_df = df[df["is_capital_area"] | df["is_border_proximate"]].copy()
    plot_df["map_category"] = "Border-proximate events"
    plot_df.loc[plot_df["is_capital_area"], "map_category"] = "Capital-area events"
    plot_df["display_fatalities"] = plot_df["fatalities"].clip(lower=0).fillna(0)
    plot_df["marker_size"] = (
        plot_df["display_fatalities"].clip(upper=400).pow(0.5) * 1.6 + 6.2
    ).round(1)

    capital_points = []
    for country_name, capital in CAPITALS_BY_COUNTRY.items():
        capital_rows = df[
            (df["country"] == country_name)
            & (
                df["location"].astype(str).eq(capital)
                | df["location"].astype(str).str.startswith(f"{capital} -")
            )
        ][["latitude", "longitude"]].drop_duplicates()
        if len(capital_rows):
            point = capital_rows.iloc[0].to_dict()
            point["country"] = country_name
            point["capital"] = capital
            capital_points.append(point)

    capitals_df = pd.DataFrame(capital_points)
    world_geojson = load_world_geojson()
    fig = go.Figure()
    fig.add_trace(
        go.Choroplethmap(
            geojson=world_geojson,
            locations=[feature["properties"]["country_name"] for feature in world_geojson["features"]],
            z=[1] * len(world_geojson["features"]),
            featureidkey="properties.country_name",
            colorscale=[[0, "#f7f3eb"], [1, "#f7f3eb"]],
            showscale=False,
            hoverinfo="skip",
            marker={"line": {"color": "#6b7280", "width": 1.0}},
            name="Country borders",
            showlegend=False,
        )
    )

    for category, color in [
        ("Capital-area events", PALETTE["capital_remote"]),
        ("Border-proximate events", PALETTE["border_battles"]),
    ]:
        sub = plot_df[plot_df["map_category"] == category].copy()
        if len(sub) > max_points_per_category:
            sub = sub.sort_values("event_date")
            step = len(sub) / max_points_per_category
            keep_idx = [int(i * step) for i in range(max_points_per_category)]
            sub = sub.iloc[keep_idx].copy()
        sub["event_date_str"] = sub["event_date"].dt.strftime("%Y-%m-%d")
        custom_cols = [
            "latitude",
            "longitude",
            "country",
            "event_date_str",
            "year",
            "event_type",
            "sub_event_type",
            "fatalities",
            "border_distance_km",
            "marker_size",
        ]
        fig.add_trace(
            go.Scattermap(
                lat=sub["latitude"].tolist(),
                lon=sub["longitude"].tolist(),
                ids=sub["event_date_str"].tolist(),
                mode="markers",
                name=category,
                customdata=sub[custom_cols].values.tolist(),
                text=sub["location"].astype(str).tolist(),
                hovertemplate=(
                    "%{text}<br>%{customdata[2]}<br>Date %{customdata[3]} (Year %{customdata[4]})"
                    "<br>%{customdata[5]} | %{customdata[6]}"
                    "<br>Fatalities %{customdata[7]}"
                    "<br>Border distance %{customdata[8]:.1f} km<extra></extra>"
                ),
                marker={
                    "size": sub["marker_size"].tolist(),
                    "color": color,
                    "opacity": 0.78,
                },
            )
        )

    if not capitals_df.empty:
        fig.add_trace(
            go.Scattermap(
                lat=capitals_df["latitude"].tolist(),
                lon=capitals_df["longitude"].tolist(),
                text=[""] * len(capitals_df),
                mode="markers",
                name="Capital cities",
                textposition="top center",
                hovertemplate="%{customdata[1]}<br>%{customdata[0]}<extra></extra>",
                customdata=capitals_df[["country", "capital"]].values.tolist(),
                marker={
                    "size": 10,
                    "symbol": "diamond",
                    "color": "#111827",
                },
                textfont={"size": 10, "color": "#111827"},
            )
        )

    fig.update_layout(
        template="plotly_white",
        height=560,
        margin={"l": 20, "r": 20, "t": 70, "b": 20},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
        title="Spatial contrast between capital-area events and border-proximate events",
        map={
            "style": "white-bg",
            "center": {"lat": 31, "lon": 31},
            "zoom": 3.05,
            "bearing": 0,
            "pitch": 0,
        },
    )
    return fig


def build_country_table(df: pd.DataFrame) -> pd.DataFrame:
    table = df.groupby("country").agg(
        capital_remote_events=("capital_remote_event", "sum"),
        border_battle_events=("border_battle_event", "sum"),
        remote_events=("is_remote", "sum"),
        total_events=("event_id_cnty", "count"),
    )
    table = table.reset_index().sort_values(
        ["capital_remote_events", "border_battle_events", "total_events"],
        ascending=[False, False, False],
    )
    return table


## Research question

Has the rise of drones and other remote attacks in the Middle East and North Africa been accompanied by a spatial shift from border-proximate conflict toward attacks on capital areas?

The analysis is descriptive rather than causal: it evaluates whether the observed spatial distribution of events is consistent with a shift toward capital targeting.


In [ ]:
headline = {
    "all_events": len(report_df),
    "border_proximate_events": int(report_df["is_border_proximate"].sum()),
    "border_proximate_share_pct": round(report_df["is_border_proximate"].mean() * 100, 2),
    "capital_area_events": int(report_df["is_capital_area"].sum()),
    "capital_area_share_pct": round(report_df["is_capital_area"].mean() * 100, 2),
    "capital_remote_events": int(report_df["capital_remote_event"].sum()),
    "capital_remote_share_of_remote_pct": round(
        report_df["capital_remote_event"].sum() / report_df["is_remote"].sum() * 100, 2
    ),
}

headline


## Figure 1: Within remote attacks

Border-proximate remote violence remains much more common than capital-area remote violence across most of the period.


In [ ]:
remote_yearly = (
    report_df.groupby("year")
    .agg(
        all_remote_events=("is_remote", "sum"),
        border_remote_events=("is_border_proximate", lambda x: int((x & report_df.loc[x.index, "is_remote"]).sum())),
        capital_remote_events=("capital_remote_event", "sum"),
    )
    .reset_index()
)
remote_yearly["border_remote_share_pct"] = (
    remote_yearly["border_remote_events"] / remote_yearly["all_remote_events"] * 100
).round(2)
remote_yearly["capital_remote_share_pct"] = (
    remote_yearly["capital_remote_events"] / remote_yearly["all_remote_events"] * 100
).round(2)

fig1 = go.Figure()
fig1.add_trace(
    go.Scatter(
        x=remote_yearly["year"],
        y=remote_yearly["border_remote_share_pct"],
        mode="lines+markers",
        name="Border-proximate share of remote attacks",
        line={"color": "#1f4e79", "width": 3.2},
    )
)
fig1.add_trace(
    go.Scatter(
        x=remote_yearly["year"],
        y=remote_yearly["capital_remote_share_pct"],
        mode="lines+markers",
        name="Capital-area share of remote attacks",
        line={"color": "#c44536", "width": 3.0},
    )
)
fig1.update_layout(
    title="Within remote attacks, border-proximate share remains above capital-area share",
    template="plotly_white",
    xaxis_title="Year",
    yaxis_title="Share (%)",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
fig1.show()


## Figure 2: Yearly all-event denominator

This figure uses yearly shares of all events. The blue segment uses non-capital border-proximate events so the stacked bars do not double-count locations that are both capital-area and border-proximate.


In [ ]:
yearly_geo = (
    report_df.assign(noncapital_border=report_df["is_border_proximate"] & ~report_df["is_capital_area"])
    .groupby("year")
    .agg(
        all_events=("event_id_cnty", "count"),
        noncapital_border_events=("noncapital_border", "sum"),
        capital_area_events=("is_capital_area", "sum"),
    )
    .reset_index()
)
yearly_geo["noncapital_border_share_pct"] = (
    yearly_geo["noncapital_border_events"] / yearly_geo["all_events"] * 100
).round(2)
yearly_geo["capital_area_share_pct"] = (
    yearly_geo["capital_area_events"] / yearly_geo["all_events"] * 100
).round(2)

fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        x=yearly_geo["year"],
        y=yearly_geo["noncapital_border_share_pct"],
        name="Non-capital border-proximate share",
        marker={"color": "#1f4e79"},
    )
)
fig2.add_trace(
    go.Bar(
        x=yearly_geo["year"],
        y=yearly_geo["capital_area_share_pct"],
        name="Capital-area share",
        marker={"color": "#c44536"},
    )
)
fig2.update_layout(
    title="Yearly all-event shares: border-proximate space remains heavier than capital space",
    template="plotly_white",
    barmode="stack",
    xaxis_title="Year",
    yaxis_title="Share of all events (%)",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
fig2.show()


## Summary tables

The summary CSV files provide the aggregate values used in the report narrative.


In [ ]:
spatial


In [ ]:
country.head(10)


## Figure 4 map

The published HTML report uses a simplified yearly map slider. The helper below builds the full Plotly map from the processed event-level data.


In [ ]:
# This can be browser-heavy because it plots event-level geography.
# Uncomment to render inside the notebook.
# build_geography_map_figure(report_df).show()


## Conclusion

The evidence supports one stable conclusion: border-linked space remains the heavier conflict geography in MENA relative to capital space. The data do not support a simple regional story in which conflict moved from borders to capitals.
